In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsentregafinal")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = (
    f"abfss://{container}@{storageName}.dfs.core.windows.net/"
    "detalle_ordenes/detalle_ordenes_raw.parquet"
)

tabla_destino = f"{catalogo}.{esquema}.detalle_ordenes"

print(f"Ruta de origen: {ruta}")
print(f"Tabla de destino: {tabla_destino}")

In [0]:
df_detalle_ordenes_read = spark.read.parquet(ruta)

In [0]:
df_detalle_ordenes_read.printSchema()

In [0]:
display(df_detalle_ordenes_read)

In [0]:
columnas_origen = set(df_detalle_ordenes_read.columns)

columnas_tabla = [
    campo.name
    for campo in spark.table(tabla_destino).schema.fields
    if campo.name.lower() != "ingestion_date"
]

columnas_faltantes = [
    columna
    for columna in columnas_tabla
    if columna not in columnas_origen
]

print("Columnas del archivo:")
print(df_detalle_ordenes_read.columns)

print("\nColumnas esperadas por la tabla Bronze:")
print(columnas_tabla)

if columnas_faltantes:
    raise ValueError(
        f"El archivo Parquet no contiene estas columnas: {columnas_faltantes}"
    )

print("\nLas columnas del archivo coinciden con la tabla Bronze.")

In [0]:
df_detalle_ordenes_final = df_detalle_ordenes_read.select(
    *[
        col(columna).cast("string").alias(columna)
        for columna in columnas_tabla
    ]
).withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
df_detalle_ordenes_final.printSchema()

display(df_detalle_ordenes_final)

In [0]:
df_detalle_ordenes_final.write.mode("overwrite").insertInto(
    tabla_destino
)

In [0]:
display(spark.table(tabla_destino))

In [0]:
cantidad_origen = df_detalle_ordenes_read.count()

cantidad_bronze = (
    spark.table(tabla_destino)
    .count()
)

print(f"Registros leídos del archivo: {cantidad_origen}")
print(f"Registros cargados en Bronze: {cantidad_bronze}")

if cantidad_origen != cantidad_bronze:
    raise ValueError(
        "La cantidad de registros del origen no coincide con Bronze."
    )

print("Carga de detalle de órdenes completada correctamente.")